**PLOTTING PREDICTIONS DISTRIBUTION**



In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import plotly.io as pio
pio.renderers.default='notebook'
import plotly.express as px
import datetime
import ast
import subprocess
import matplotlib.colors as mcolors
import seaborn as sns
import plotly.express as px

In [2]:
COLLOR_PALLET = {
            'Other human': '#2986cc', # BLUE
            'Electro-mechanical': '#cc0000', # RED
            'Voice': '#6aa84f', #  green 6aa84f
            'Motorised transport': '#ffa500', # orange
            'Geonature': '#8e7cc3', # PURPLE
            'Animal': '#9b5f00', # BROWN
            'Music': '#d172a4', # PINK
            'Background': '#000000', # BLACK
            'Other Sounds': '#c9d631', # yellow
            'Social/communal': '#d8cbf8', # Light purple
            'Human movement': '#40b674', # light green 40b674
        }

def extract_location(file_path):
    file_name = os.path.basename(file_path)
    # print(f"File name: {file_name}")
    # Split the file name by the underscore
    file_name = file_name.split("_")[2]
    # print(f"File name after split: {file_name}")
    return file_name
    
def postprocessing_df(df_preds):
    # df_preds.drop(columns=['classes_custom', 'probabilities_custom', 'sum_probs_custom', 'sum_probs_original'],inplace=True)
    df_preds.rename(columns={'datetime':'filenames', 'classes_original':'classes'}, inplace=True)
    df_preds['classes'] = df_preds['classes'].apply(ast.literal_eval)
    
    return df_preds

def insert_time(df_preds):
    df_preds['date'] = pd.to_datetime(df_preds['filenames'])

    df_preds['year'] = df_preds.apply(lambda x: x['date'].year, axis= 1)
    df_preds['month'] = df_preds.apply(lambda x: x['date'].month, axis= 1)
    df_preds['day'] = df_preds.apply(lambda x: x['date'].day, axis= 1)

    df_preds['day_name'] = df_preds.apply(lambda x: x['date'].day_name(), axis= 1)
    df_preds['weekday'] = df_preds.apply(lambda x: x['date'].weekday(), axis= 1)
    
    df_preds['hour'] = df_preds.apply(lambda x: x['date'].hour, axis= 1)
    df_preds['hour_min'] = df_preds.apply(lambda x: str(x['date'].hour) + '_' + str(x['date'].minute),axis=1)
    
    df_preds['time'] = df_preds.apply(lambda x: x['date'].time(),axis=1)
    
    return df_preds

def index_and_explode(df):
    df.set_index('time',inplace=True)
    df = df.explode('classes')
    df['display_name'] = df['classes']
    df['number'] = 1
    
    return df

def merge_dataframes(df, union):
    df = df.merge(union,how='left',on='display_name')
    
    return df

def remove_label(df, column, label):
    df_no_label = df[df[column] != label]
    return df_no_label, label

# sort by time column called datetime
def sort_df(df):
    df.sort_values(by=['datetime'], inplace=True)
    return df

def output_dir(path: str):
    # get the abs path and remove the last element
    path = os.path.abspath(path).split("\\")[:-2]
    path = "/".join(path)
    
    visualization_dir = path + "/Visualizations"
    os.makedirs(visualization_dir, exist_ok=True)
    
    return visualization_dir

# get the last git tag version
def list_git_tags():
    try:
        tags = tags = subprocess.check_output(["git", "tag"]).strip().decode()
        return tags.split('\n')
    except subprocess.CalledProcessError:
        return None
    
def select_tag(tags):
    for i, tag in enumerate(tags):
        print(f"{i}: {tag}")
    choice = int(input("Select the tag to use: "))
    tag_selected = tags[choice]
    # replace "." with "_" to be able to use it as a file name
    tag_selected = tag_selected.replace(".", "_")
        
    return tag_selected

def get_stable_version():
    tags = list_git_tags()
    # get the latest stable version
    tag_selected = tags[-1]
    print(f"Latest stable version with period: {tag_selected}")
    # replace "." with "_" to be able to use it as a file name
    tag_selected = tag_selected.replace(".", "_")
    
    print(f"Latest stable version with underscore: {tag_selected}")
    
    return tag_selected

stable_version = get_stable_version()

Latest stable version with period: v1.1
Latest stable version with underscore: v1_1


In [3]:
# df_preds = pd.read_csv(f'avg_predictions_{abrev}.csv', converters={'classes': eval})
# yammnet_class_map = 'yamnet_class_map_rev_FINAL.csv'
yammnet_class_map = r"C:\Users\scjaa\Documents\GitHubRepos\AAC\AI_Model\Urban_Model\taxonomy_mapping\yamnet_class_AAC_v1_0.csv"
union = pd.read_csv(yammnet_class_map,sep=';')

path_input = input("Enter the path to the csv file: ")
path_input = path_input.replace('"', '')

# make visualization directory
visualization_dir = output_dir(path_input)

# get the location
title = extract_location(path_input)

# get the csv
df_preds = pd.read_csv(path_input)

df_preds

,file,datetime,classes_original,probabilities_original
0,20240313_114835.WAV,2024-03-13 11:48:35,"['Speech', 'Vehicle', 'Inside, large room or h...","[0.33249143, 0.06553795, 0.021650873]"
1,20240313_120340.WAV,2024-03-13 12:03:40,"['Speech', 'Narration, monologue', 'Choir']","[0.5096697, 0.030688513, 0.025130635]"
2,20240313_121845.WAV,2024-03-13 12:18:45,"['Speech', 'Vehicle', 'Narration, monologue']","[0.38610014, 0.050903738, 0.02035032]"
3,20240313_123350.WAV,2024-03-13 12:33:50,"['Speech', 'Vehicle', 'Boat, Water vehicle']","[0.37306717, 0.057075504, 0.021135075]"
4,20240313_124855.WAV,2024-03-13 12:48:55,"['Speech', 'Vehicle', 'Narration, monologue']","[0.3861818, 0.065761894, 0.02198071]"
...,...,...,...,...
563,20240319_113330.WAV,2024-03-19 11:33:30,"['Vehicle', 'Motor vehicle (road)', 'Motorcycle']","[0.5040235, 0.17247647, 0.14509241]"
564,20240319_114835.WAV,2024-03-19 11:48:35,"['Vehicle', 'Motor vehicle (road)', 'Car']","[0.4497974, 0.15203646, 0.119432285]"
565,20240319_120340.WAV,2024-03-19 12:03:40,"['White noise', 'Rustle', 'Snake']","[0.16818288, 0.12258033, 0.12049646]"
566,20240319_121845.WAV,2024-03-19 12:18:45,"['Snake', 'White noise', 'Rustle']","[0.18830791, 0.14690547, 0.13402537]"


In [4]:
df_preds = sort_df(df_preds)
df_preds = postprocessing_df(df_preds)
df_preds = insert_time(df_preds)

df = df_preds.copy()

df = index_and_explode(df)
df = merge_dataframes(df, union)

start_date = df.sort_values('date')['date'].iloc[0]
end_date = df.sort_values('date')['date'].iloc[-1]
print('Inicio Monitoreo: {} \nFinal Monitoreo: {}'.format(start_date,end_date))

# unique_classes = set(df['classes'].unique())
# df_no_Nature, label = remove_label(df, 'Brown_Level_2', 'Nature')

Inicio Monitoreo: 2024-03-13 11:48:35 
Final Monitoreo: 2024-03-19 12:33:50


## **PLOTTING GLOBAL DASH BOARD**

In [5]:
# display_name, Brown_Level_1, Brown_Level_2, Brown_Level_3
display_name = 'display_name'
brown_2 = 'Brown_Level_2'
brown_3 = 'Brown_Level_3'

# set what detail level to use
deepth = brown_2

In [7]:
df

,file,filenames,classes,probabilities_original,date,year,month,day,day_name,weekday,...,mid,iso_taxonomy,Brown_Level_1,Brown_Level_2,Brown_Level_3,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,20240313_114835.WAV,2024-03-13 11:48:35,Speech,"[0.33249143, 0.06553795, 0.021650873]",2024-03-13 11:48:35,2024,3,13,Wednesday,2,...,/m/09x0r,Voice,NaN,Voice,speech,NaN,NaN,NaN,NaN,NaN
1,20240313_114835.WAV,2024-03-13 11:48:35,Vehicle,"[0.33249143, 0.06553795, 0.021650873]",2024-03-13 11:48:35,2024,3,13,Wednesday,2,...,/m/07yv9,Roadway traffic,NaN,Motorised transport,roadway traffic,NaN,NaN,NaN,NaN,NaN
2,20240313_114835.WAV,2024-03-13 11:48:35,"Inside, large room or hall","[0.33249143, 0.06553795, 0.021650873]",2024-03-13 11:48:35,2024,3,13,Wednesday,2,...,/t/dd00126,Background,NaN,Background,Background people,NaN,NaN,NaN,NaN,NaN
3,20240313_120340.WAV,2024-03-13 12:03:40,Speech,"[0.5096697, 0.030688513, 0.025130635]",2024-03-13 12:03:40,2024,3,13,Wednesday,2,...,/m/09x0r,Voice,NaN,Voice,speech,NaN,NaN,NaN,NaN,NaN
4,20240313_120340.WAV,2024-03-13 12:03:40,"Narration, monologue","[0.5096697, 0.030688513, 0.025130635]",2024-03-13 12:03:40,2024,3,13,Wednesday,2,...,/m/02qldy,Voice,NaN,Voice,speech,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1699,20240319_121845.WAV,2024-03-19 12:18:45,White noise,"[0.18830791, 0.14690547, 0.13402537]",2024-03-19 12:18:45,2024,3,19,Tuesday,1,...,/m/0chx_,Background,NaN,Other Sounds,noise,NaN,NaN,NaN,NaN,NaN
1700,20240319_121845.WAV,2024-03-19 12:18:45,Rustle,"[0.18830791, 0.14690547, 0.13402537]",2024-03-19 12:18:45,2024,3,19,Tuesday,1,...,/m/07qwyj0,Fx,NaN,Other Sounds,Other Sounds,NaN,NaN,NaN,NaN,NaN
1701,20240319_123350.WAV,2024-03-19 12:33:50,Vehicle,"[0.21137072, 0.10038923, 0.052070245]",2024-03-19 12:33:50,2024,3,19,Tuesday,1,...,/m/07yv9,Roadway traffic,NaN,Motorised transport,roadway traffic,NaN,NaN,NaN,NaN,NaN
1702,20240319_123350.WAV,2024-03-19 12:33:50,Speech,"[0.21137072, 0.10038923, 0.052070245]",2024-03-19 12:33:50,2024,3,19,Tuesday,1,...,/m/09x0r,Voice,NaN,Voice,speech,NaN,NaN,NaN,NaN,NaN


In [8]:
fig = px.treemap(df, 
                 path=[deepth, 'classes'], 
                 values='number',
                 color=deepth,  #for coloring
                 color_discrete_map=COLLOR_PALLET 
                )

fig.update_layout(title=f'{title} Global | Urban Model | From {df["date"].iloc[0]} to {df["date"].iloc[-1]}')
fig.show()

# fig.write_html(f"{visualization_dir}/{title}_Global_UrbanModel_{stable_version}.html")

In [ ]:
# fig = px.treemap(df_no_Nature, 
#                     path=['Brown_Level_2', 'classes'], 
#                     values='number'
#                 )
# fig.update_layout(title=f'{title} Global Sin {label} | {df["date"].iloc[0]} - {df["date"].iloc[-1]}')

# # save figure
# fig.write_html(f"{visualization_dir}/{title}_Global_Sin_{label}_{stable_version}.html")

# fig.show()


**PLOTTING EACH DAY**

In [ ]:
for day in df['day'].unique():
    day_df = df[df['day'] == day]
    fig = px.treemap(day_df, 
                     path=[deepth, 'classes'], 
                     values='number',
                     color=deepth,
                     color_discrete_map=COLLOR_PALLET
                    )
    fig.update_layout(title=f'{title} | Urban Model | {day_df["year"].iloc[0]}-{day_df["month"].iloc[0]}-{day} - {day_df["day_name"].iloc[0]}')
    
    # save figure
    fig.write_html(f"{visualization_dir}/{title}_Day_{day}_Global_UrbanModel_{stable_version}.html")

    fig.show()

# PLOTTING FOR EACH 4 HOURS STARTING AT 07:00 AM

In [ ]:
for column in df.columns:
    if 'Unnamed' in column:
        df.drop(columns=column, inplace=True)

In [ ]:
# set the datetime column as index
df.set_index('filenames', inplace=True)

In [ ]:
# for each 4-hour interval starting at 07:00 am
for day in df['day'].unique():
    day_df = df[df['day'] == day]
    print(f"Day: {day}")
    # print hour
    print(f"Hour: {day_df['hour'].iloc[0]}")

    for hour in range(7, 24, 4):  # 7, 11, 15, 19, 23
        # Select data between 'hour' and 'hour + 4', wrapping around midnight if necessary
        next_hour = hour + 4
        print(f"Hour: {hour}")
        print(f"Next hour: {next_hour}")
        
        if next_hour > 23:
            hour_df = df[(df['hour'] >= hour) | (df['hour'] < next_hour - 24)]
        else:
            hour_df = df[(df['hour'] >= hour) & (df['hour'] < next_hour)]

        if not hour_df.empty:
            print(f"Time Interval: {hour}:00 to {next_hour}:00")

            fig = px.treemap(hour_df, 
                            path=[deepth, 'classes'], 
                            values='number',
                            color=deepth,
                            color_discrete_map=COLLOR_PALLET
                            )
            fig.update_layout(title=f'{title} | Urban Model | {hour_df["year"].iloc[0]}-{hour_df["month"].iloc[0]}-{hour_df["day"].iloc[0]} - {hour_df["day_name"].iloc[0]} - {hour}:00 to {next_hour}:00')
            
            fig.show()

Day: 22
Hour: 9
Hour: 7
Next hour: 11
Time Interval: 7:00 to 11:00


Hour: 11
Next hour: 15
Time Interval: 11:00 to 15:00


Hour: 15
Next hour: 19
Time Interval: 15:00 to 19:00


Hour: 19
Next hour: 23
Time Interval: 19:00 to 23:00


Hour: 23
Next hour: 27
Time Interval: 23:00 to 27:00


...

Day: 27
Hour: 0
Hour: 7
Next hour: 11
Time Interval: 7:00 to 11:00


Hour: 11
Next hour: 15
Time Interval: 11:00 to 15:00


Hour: 15
Next hour: 19
Time Interval: 15:00 to 19:00


Hour: 19
Next hour: 23
Time Interval: 19:00 to 23:00


Hour: 23
Next hour: 27
Time Interval: 23:00 to 27:00


In [ ]:
for day in df_no_Nature['day'].unique():
    day_df = df_no_Nature[df_no_Nature['day'] == day]
    fig = px.treemap(df_no_Nature, 
                     path=['Brown_Level_2', 'classes'], 
                     values='number'
                    )
    fig.update_layout(title=f'{title} Sin {label} | {day_df["year"].iloc[0]}-{day_df["month"].iloc[0]}-{day}')

    fig.show()

**PLOTTING SPECIFIC TIME SPANS**

In [ ]:
START_TIME_SPECIFIC = "2022-10-29 13:00:00"
END_TIME_SPECIFIC = "2022-10-29 15:15:00"

In [ ]:
start_time = pd.Timestamp(START_TIME_SPECIFIC)
end_time = pd.Timestamp(END_TIME_SPECIFIC)

filtered_df = df[(df['date'] >= start_time) & (df['date'] <= end_time)]

filtered_df['number'] = 1
fig = px.treemap(filtered_df, 
                    path=['Brown_Level_2', 'classes'], 
                    values='number',
                    color='Brown_Level_2',
                    color_discrete_map=COLLOR_PALLET
                )
fig.update_layout(title=f'{title} | {start_time} - {end_time} ')

# save fuiture
fig.write_html(f"{visualization_dir}/{title}_{START_TIME_SPECIFIC}_{END_TIME_SPECIFIC}_{stable_version}.html")

fig.show()

OSError: [Errno 22] Invalid argument: '\\\\192.168.205.117\\AAC_Server\\INDUSTRIA\\23603_GRANTIERRA\\5-Resultados\\Audiomoth2\\URBAN_Model\\Visualizations\\GRANTIERRA-P2_2022-10-29 13:00:00_2022-10-29 15:15:00_v1_0.html'

In [ ]:
start_time = pd.Timestamp(START_TIME_SPECIFIC)
end_time = pd.Timestamp(END_TIME_SPECIFIC)
filtered_df = df_no_Nature[(df['date'] >= start_time) & (df['date'] <= end_time)]

filtered_df['number'] = 1
fig = px.treemap(filtered_df, 
                    path=['Brown_Level_2', 'classes'], 
                    values='number',
                    color='Brown_Level_2',
                    color_discrete_map=COLLOR_PALLET
                )
fig.update_layout(title=f'{title} Sin {label} | {start_time} - {end_time} ')

# save figure
fig.write_html(f"{visualization_dir}/{title}_{START_TIME_SPECIFIC}_{END_TIME_SPECIFIC}_Sin_{label}.html")

fig.show()

/tmp/ipykernel_16501/2334339269.py:3: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.

/tmp/ipykernel_16501/2334339269.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
start_time = pd.Timestamp("2022-10-29 20:00:00")
end_time = pd.Timestamp("2022-10-29 21:00:00")
filtered_two_df = df[(df['date'] >= start_time) & (df['date'] <= end_time)]

filtered_df = df[(df['date'] >= start_time) & (df['date'] <= end_time)]

filtered_df['number'] = 1
fig = px.treemap(filtered_df, 
                    path=['Brown_Level_2', 'classes'], 
                    values='number',
                    color='Brown_Level_2',
                    color_discrete_map=COLLOR_PALLET
                )
fig.update_layout(title=f'{title} | {start_time} - {end_time} ')
fig.show()

/tmp/ipykernel_16501/3784541672.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
start_time = pd.Timestamp("2022-10-20 13:30:00")
end_time = pd.Timestamp("2022-10-20 14:50:00")
filtered_two_df = df_no_Nature[(df['date'] >= start_time) & (df['date'] <= end_time)]

filtered_two_df['number'] = 1
fig = px.treemap(filtered_two_df, 
                    path=['Brown_Level_2', 'classes'], 
                    values='number',
                    color='Brown_Level_2',
                    color_discrete_map=COLLOR_PALLET
                )
fig.update_layout(title=f'{title} Sin {label} | {start_time} - {end_time}')
fig.show()

/tmp/ipykernel_16501/2926250780.py:3: UserWarning:

Boolean Series key will be reindexed to match DataFrame index.



In [ ]:
start_time = pd.Timestamp("2022-10-29 21:20:00")
end_time = pd.Timestamp("2022-10-29 23:55:00")
filtered_df = df[(df['date'] >= start_time) & (df['date'] <= end_time)]

filtered_df['number'] = 1
fig = px.treemap(filtered_df, 
                    path=['Brown_Level_2', 'classes'], 
                    values='number',
                    color='Brown_Level_2',
                    color_discrete_map=COLLOR_PALLET
                )
fig.update_layout(title=f'{title} | {start_time} - {end_time} ')
fig.show()

/tmp/ipykernel_16501/245140038.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
start_time = pd.Timestamp("2022-11-09 09:45:00")
end_time = pd.Timestamp("2022-11-09 12:00:00")
filtered_df = df[(df['date'] >= start_time) & (df['date'] <= end_time)]

filtered_df['number'] = 1
fig = px.treemap(filtered_df, 
                    path=['Brown_Level_2', 'classes'], 
                    values='number',
                    color='Brown_Level_2',
                    color_discrete_map=COLLOR_PALLET
                )
fig.update_layout(title=f'{title} | {start_time} - {end_time} ')
fig.show()

/tmp/ipykernel_16501/3523570605.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
start_time = pd.Timestamp("2022-11-14 11:00:00")
end_time = pd.Timestamp("2022-11-14 13:00:00")
filtered_df = df[(df['date'] >= start_time) & (df['date'] <= end_time)]

filtered_df['number'] = 1
fig = px.treemap(filtered_df, 
                    path=['Brown_Level_2', 'classes'], 
                    values='number',
                    color='Brown_Level_2',
                    color_discrete_map=COLLOR_PALLET
                )
fig.update_layout(title=f'{title} | {start_time} - {end_time} ')
fig.show()

/tmp/ipykernel_16501/408549486.py:5: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

